In [ ]:

# Ensure the current Python interpreter's (active .venv) site-packages are on sys.path, then import numpy
import sys, site
site.addsitedir(f"{sys.prefix}/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages")
import numpy as np

In [13]:

!pip install opencv-python


  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 2.1 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/12.4 MB 1.5 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.4 MB 1.4 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/12.4 MB 822.3 kB/s eta 0:00:14
   ------- ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.1 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.5.1 which is incompatible.


In [ ]:
import cv2
import numpy

In [ ]:


# Access the camera feed (0 for default webcam)
cap = cv2.VideoCapture(0)

frames = []
gap = 5  # Difference between current frame and the 5th previous frame [4]
count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame to grayscale for processing [5]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frames.append(gray)

    # Maintain a sliding window of frames based on the gap [6, 7]
    if len(frames) > (gap + 1):
        frames.pop(0)

    # Begin detection logic once the buffer is full [2]
    if len(frames) == (gap + 1):
        # Calculate absolute difference between the first and last frame in buffer [2, 8]
        diff = cv2.absdiff(frames, frames[-1])
        
        # Apply thresholding to ignore minor pixel flickers [8]
        _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
        
        # Find contours (clusters of moving pixels) [9, 10]
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        motion_detected = False
        for c in contours:
            # Only consider contours with a significant area (e.g., > 500 pixels) [3, 11]
            if cv2.contourArea(c) < 500:
                continue
            
            motion_detected = True
            # Draw a green bounding rectangle around the detected motion [3, 12]
            (x, y, w, h) = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        if motion_detected:
            cv2.putText(frame, "MOTION DETECTED", (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2) [13]

    # Display the result [14]
    cv2.imshow("Burglar Alarm System", frame)
    
    # Exit on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'cv2' is not defined

In [1]:
import cv2

# If CELL 2 still errors later, also fix the diff line there:
# diff = cv2.absdiff(frames[0], frames[-1])

In [ ]:
from pathlib import Path

img=cv2.imread("./building.jpg")  # 
if img is None:
    p = Path.cwd() / "building.jpg"
    if not p.exists():
        # try to find any file named like 'building' in the current directory
        candidates = list(Path.cwd().glob("building.*")) + list(Path.cwd().glob("building*.*"))
        if candidates:
            p = candidates[0]
        else:
            print(f"Error: 'building.jpg' not found in {Path.cwd()}")
            img = None
    # attempt to read (overwrite img)
    if p.exists():
        img = cv2.imread(str(p))
        if img is None:
            print(f"Error: Failed to read image at {p}")
    else:
        # Use project folder as base so paths resolve relative to the notebook/project directory
        base = Path.cwd()
        BASE_DIR = base  # available for other cells to build paths from the project folder
        # try direct filename in project folder
        alt = base / p.name
        if alt.exists():
            img = cv2.imread(str(alt))
            if img is None:
                print(f"Error: Failed to read image at {alt}")
            else:
                p = alt
        else:
            # fallback: search common patterns in project folder
            candidates = list(base.glob("building.*")) + list(base.glob("building*.*"))
            if candidates:
                p = candidates[0]
                img = cv2.imread(str(p))
                if img is None:
                    print(f"Error: Failed to read image at {p}")
            else:
                print(f"Error: 'building.jpg' not found in {base}")

Error: 'building.jpg' not found in C:\Users\Asim Shah


In [3]:
cv2.imshow("Building", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

error: OpenCV(5.0.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:951: error: (-215:Assertion failed) size.width>0 && size.height>0 in function 'cv::imshow'
